<a href="https://colab.research.google.com/github/PratikshitSingh/AI-Model-Evaluation---Training/blob/main/1_AdvancedAIModelEval_TraceBasedEvalSetup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Trace-Based Evaluation Setup

## Module 01 — Modern AI Evaluation Foundations

---

## Overview

In this lab, you will configure **Langfuse** to capture traces from LLM interactions and explore how trace data enables evaluation that goes far beyond inspecting model outputs alone.

By the end of this lab you will have:
- A working Langfuse tracing setup
- Multiple traced LLM interactions
- Hands-on experience exploring trace data
- Evaluation scores attached to traces

## Learning Objectives

1. Configure Langfuse tracing for an LLM application
2. Capture traces including prompts, responses, latency, and token usage
3. Explain what trace-based evaluation captures that output-only evaluation misses
4. Design evaluation criteria using trace data
5. Attach evaluation scores to traces programmatically

## Prerequisites

- Python proficiency
- Basic understanding of LLM APIs (prompts, completions)
- Familiarity with ML evaluation concepts (accuracy, precision, recall)

## Estimated Time

**15–20 minutes** hands-on

## Scenario

You are an ML engineer tasked with setting up an evaluation framework for a customer-facing Q&A assistant. The assistant answers questions about a company's product documentation. Management wants to move beyond "does it sound right?" to rigorous, traceable evaluation. Your first step: get tracing working and understand what data it gives you.

---

## 🔑 Understanding Langfuse and Traces: A Beginner's Guide

Before diving into the lab, let's understand the foundational concepts you'll be working with.

### What is Langfuse?

**Langfuse** is an open-source **observability and analytics platform** specifically designed for LLM (Large Language Model) applications. Think of it as a "debugger" or "monitoring dashboard" for AI applications.

When you build an application that uses LLMs (like ChatGPT, Claude, or open-source models), you need to:
- See what prompts are being sent
- Track what responses come back
- Measure how long things take
- Identify errors and issues
- Evaluate quality over time

Langfuse gives you all of this visibility in one place.

**Analogy:** If you've ever used logging tools like CloudWatch or Datadog for traditional applications, Langfuse is the equivalent for LLM applications — but with AI-specific features like token tracking and evaluation scoring.

### What is a Trace?

A **trace** is a complete record of a single interaction with your LLM application. It captures everything that happened from the moment a user asked a question until they received an answer.

A trace typically contains:

| Component | What It Captures | Why It Matters |
|-----------|------------------|----------------|
| **Input** | The user's question or prompt | Understand what users are asking |
| **Output** | The model's response | See what your AI actually said |
| **Latency** | How long the response took | Monitor user experience and performance |
| **Token Usage** | Number of tokens consumed | Track costs (you pay per token!) |
| **Model Info** | Which model was used (e.g., gpt-4o-mini) | Compare model performance |
| **Metadata** | Custom tags and labels you add | Filter and organize traces |
| **Timestamps** | When each step happened | Debug timing issues |

### Trace Hierarchy: Traces, Spans, and Generations

Traces have a **hierarchical structure**. This matters because real LLM applications often involve multiple steps:

```
TRACE (the overall interaction)
 └── SPAN (a logical operation, like "process user request")
      └── GENERATION (an actual LLM API call)
      └── GENERATION (another LLM API call, if needed)
 └── SPAN (another operation, like "retrieve documents")
```

- **Trace**: The parent container for an entire user interaction
- **Span**: A logical grouping of work (e.g., retrieval, validation, formatting)
- **Generation**: A specific call to an LLM that produces text

### A Simple Example: What a Trace Looks Like

Imagine a user asks your Q&A assistant: *"What is the maximum file upload size?"*

Here's what gets captured in the trace:

```
📦 TRACE: qa-assistant-simple-factual
├── trace_id: "abc123-def456"
├── timestamp: "2026-03-28T10:15:30Z"
├── input: "What is the maximum file upload size?"
├── output: "The maximum file upload size is 50 MB for free-tier..."
├── metadata: { source: "product-docs-assistant" }
│
└── 📄 GENERATION: llm-completion
    ├── model: "gpt-4o-mini"
    ├── input_tokens: 12
    ├── output_tokens: 24
    ├── latency: 0.8 seconds
    └── simulated: true
```


What's missing?

👉 The SPAN layer is skipped

Normally you'd expect something like:

```
📦 TRACE: qa-assistant-simple-factual
│
└── 🔹 SPAN: answer-question-step        👈 (THIS is missing)
    │
    └── 📄 GENERATION: llm-completion
```

🤔 Why is there no span here?

Because this is a simple, single-step flow.

In many systems:

If you only have one LLM call
and no meaningful intermediate steps

👉 The system implicitly treats the generation as the step, and doesn’t require a separate span



Every piece of information above is captured automatically when you instrument your code with Langfuse.

### Why Do We Use Traces for Evaluation?

**The Problem with Output-Only Evaluation:**
If you only look at what an LLM said (the output), you miss critical information:
- Was it fast or slow?
- Did it use way too many tokens (expensive)?
- Which model version produced this response?
- Were there intermediate steps that failed?

**Traces Enable Better Evaluation:**
With traces, you can:
1. **Attach evaluation scores** directly to specific interactions
2. **Filter and compare** responses across different conditions
3. **Trend analysis** — see if quality is improving or degrading over time
4. **Debug failures** — when something goes wrong, you have full context
5. **Cost analysis** — understand which interactions are expensive

### Quick Vocabulary Reference

| Term | Definition |
|------|------------|
| **Langfuse** | Open-source LLM observability platform |
| **Trace** | A complete record of one user interaction |
| **Span** | A logical step or operation within a trace |
| **Generation** | A specific LLM API call that produces output |
| **Observation** | Generic term for spans and generations (things inside a trace) |
| **Score** | An evaluation metric attached to a trace (e.g., correctness: 0.9) |
| **Flush** | Sending all pending trace data to Langfuse's servers |

---

Now you're ready to start the hands-on lab! You'll configure Langfuse, create traced interactions, and attach evaluation scores.

---

## Task 1: Environment Setup and Dependencies

Here’s the exact **updated Task 1 text** you can copy/paste:

---

# Task 1: Set Up Your Langfuse Account and Project

In this exercise, you will create your own Langfuse account and project. Each person will work in their own isolated environment to keep traces clean and easy to debug.

## Step 1: Create a Langfuse Account

1. Go to [https://cloud.langfuse.com](https://cloud.langfuse.com)
2. Sign up using Google, GitHub, or email

## Step 2: Create an Organization

1. After logging in, you will land on the **Organizations** page
2. Click **+ New Organization**
3. Name it something like:

   * `advanced-model-course`
   * `llm-lab`
   * `personal-ai-projects`

## Step 3: Create a Project

1. Enter your organization
2. Click **New Project**
3. Name it something like:

   * `langfuse-tracing-demo`
   * `rag-eval-lab`
   * `agent-workflow`

💡 A project represents one application or system you want to observe (e.g., a chatbot, RAG pipeline, or agent workflow).

## Step 4: Get Your API Keys

1. Inside your project, go to **Settings → API Keys -> Create new API Keys**.  For the optional note you can just call it "class key"

2. Copy the following:

   * Public Key (`pk-lf-...`)
   * Secret Key (`sk-lf-...`)

⚠️ Important:

* The secret key is only shown once — save it securely
* Do NOT share your keys with others

## Step 5: Configure in Your Notebook

You will use these keys in the next step to connect your notebook to Langfuse.

## Why we do this

* Each student has isolated traces (no noise from others)
* Easier debugging and experimentation
* Matches real-world production setups



### Goal
Install the required packages so that Langfuse tracing works in this Colab environment.

### Steps
Run the cell below to install all dependencies. This may take a minute.

In [ ]:
%%capture
# ============================================================================
# PACKAGE INSTALLATION
# ============================================================================
# This cell installs the Python packages required for this lab.
# The %%capture magic suppresses output to keep the notebook clean.
#
# Packages being installed:
#   - langfuse: The official Langfuse Python SDK for tracing and observability.
#               This is how our code communicates with Langfuse's servers.
#   - openai:   The OpenAI Python SDK. While we use simulated responses in this
#               lab, we include this for compatibility with real-world usage.
#
# The -U flag ensures we get the latest versions of both packages.
# ============================================================================

!pip install -U langfuse openai

In [ ]:
# ============================================================================
# PYTHON STANDARD LIBRARY IMPORTS
# ============================================================================
# These are built-in Python modules we'll use throughout the lab.
#
#   - os:       Access environment variables (for storing API keys securely)
#   - json:     Parse and create JSON data structures (trace data is often JSON)
#   - time:     Add delays between operations (give Langfuse time to process)
#   - uuid:     Generate unique identifiers (useful for trace correlation)
#   - datetime: Work with timestamps (traces are time-stamped)
#   - timezone: Ensure consistent UTC timestamps across all traces
# ============================================================================

import os
import json
import time
import uuid
from datetime import datetime, timezone

### Checkpoint ✅
If the cells above ran without errors, your environment is ready. You should see no output (installs are captured) and the imports should complete silently.

---

## Task 2: Configure Langfuse Connection

### Goal
Initialize the Langfuse client and verify it can communicate with the Langfuse backend.

### Steps

1. Set your Langfuse credentials using your own project keys.
2. Run the configuration cell below.
3. Verify the connection succeeds.

> **Note:** Current Langfuse SDK docs use `LANGFUSE_BASE_URL` for the host setting. We set both `LANGFUSE_BASE_URL` and `LANGFUSE_HOST` below for compatibility.


In [ ]:
# ============================================================================
# LANGFUSE CONFIGURATION - SETTING UP YOUR CONNECTION
# ============================================================================
# Langfuse uses environment variables to authenticate and connect to your
# project. This is a security best practice — we don't hardcode secrets.
#
# THREE REQUIRED SETTINGS:
#
# 1. LANGFUSE_PUBLIC_KEY (pk-lf-...)
#    - Identifies your project
#    - Safe to expose in client-side code (but still keep private)
#    - You get this from: Langfuse Dashboard → Settings → API Keys
#
# 2. LANGFUSE_SECRET_KEY (sk-lf-...)
#    - Authenticates your requests
#    - NEVER share this publicly (treat like a password)
#    - You get this from: Langfuse Dashboard → Settings → API Keys
#    - ⚠️ Only shown once when created — save it immediately!
#
# 3. LANGFUSE_HOST / LANGFUSE_BASE_URL
#    - The Langfuse server to send data to
#    - US region: https://us.cloud.langfuse.com
#    - EU region: https://cloud.langfuse.com
#    - Self-hosted: your own server URL
# ============================================================================

# --- Option A: Set credentials directly (quickest for classroom labs) ---
# --- Option B: Load from Colab Secrets or environment variables (production) ---

# ⚠️ REPLACE THESE with your own Langfuse project keys from Task 1!
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-ea9b60d6-68a8-410e-ba49-39ca33d0534d"   # <-- Your public key
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-98300f5c-7cc5-4faf-84df-b56c195f5b74"   # <-- Your secret key

# Set the Langfuse cloud host URL (use US or EU depending on your account region)
os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com"

# LANGFUSE_BASE_URL is the newer setting name; we set both for compatibility
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"

# Ensure backward compatibility with older examples that use LANGFUSE_HOST
os.environ["LANGFUSE_HOST"] = os.environ["LANGFUSE_BASE_URL"]

In [ ]:
# ============================================================================
# INITIALIZE THE LANGFUSE CLIENT
# ============================================================================
# The Langfuse client is the main object you use to interact with Langfuse.
# It handles:
#   - Authentication with your API keys
#   - Sending trace data to Langfuse servers
#   - Querying traces and scores
#   - Managing the connection lifecycle
#
# get_client() reads from the environment variables we set above and returns
# a configured client instance. This is the recommended way to create a client
# in the current SDK version.
#
# auth_check() verifies that:
#   - Your API keys are valid
#   - The host URL is reachable
#   - You have permission to send traces to this project
# ============================================================================

from langfuse import get_client

# Create a Langfuse client instance using the environment variables we configured
langfuse = get_client()

# Verify credentials are valid and we can connect to Langfuse
# This will raise an exception if authentication fails
langfuse.auth_check()

print("✅ Langfuse connection successful!")

✅ Langfuse connection successful!


### Checkpoint ✅
You should see `✅ Langfuse connection successful!` above. If you see an authentication error, double-check your API keys and confirm your base URL matches your Langfuse region.

---

## Task 3: Capture Traces from LLM Interactions

### Goal
Create multiple traced LLM interactions that simulate a product Q&A assistant. We'll use **simulated responses** so the lab works reliably without requiring an OpenAI API key.

Each interaction will be fully traced in Langfuse — capturing the prompt, the response, latency, token counts, and metadata.

### Why simulate?
In a classroom setting, simulated calls ensure everyone gets consistent results, avoids API rate limits, and removes the need for individual OpenAI keys. The tracing mechanics are identical to real calls.

> **SDK note:** The current Langfuse Python SDK uses `start_as_current_observation(...)` rather than older examples that used `trace()` or `start_trace()`.


In [ ]:
# ============================================================================
# SIMULATED Q&A INTERACTIONS - OUR TEST DATASET
# ============================================================================
# In a real application, these would be live user questions sent to an LLM.
# For this lab, we simulate responses to ensure:
#   - Consistent results across all students
#   - No dependency on OpenAI API keys
#   - Predictable data for learning and debugging
#
# IMPORTANT: The tracing mechanics are IDENTICAL whether using simulated or
# real LLM responses. Langfuse doesn't know the difference!
#
# Each interaction dictionary contains:
#   - name:         A descriptive identifier for this type of question
#   - user:         The question the user asked (the input to our assistant)
#   - assistant:    The response our assistant would give (the output)
#   - latency:      Simulated response time in seconds (mimics real API calls)
#   - input_tokens: Number of tokens in the user's question
#   - output_tokens: Number of tokens in the assistant's response
#   - model:        Which LLM model supposedly generated this response
#
# We've chosen 4 diverse scenarios that represent real-world evaluation challenges:
#   1. simple_factual     - A straightforward question with a clear answer
#   2. multi_step_reasoning - Requires understanding context and giving multi-part advice
#   3. ambiguous_question - User intent is unclear (delete files? delete account?)
#   4. out_of_scope       - Question is outside the assistant's knowledge domain
# ============================================================================

QA_INTERACTIONS = [
    {
        "name": "simple_factual",
        # A direct, unambiguous question with a factual answer
        "user": "What is the maximum file upload size?",
        "assistant": "The maximum file upload size is 50 MB for free-tier users and 500 MB for enterprise users.",
        "latency": 0.8,         # Fast response - simple lookup
        "input_tokens": 12,     # Short question = few tokens
        "output_tokens": 24,    # Concise answer
        "model": "gpt-4o-mini",
    },
    {
        "name": "multi_step_reasoning",
        # User has a problem that requires diagnosing and providing steps
        "user": "I upgraded to enterprise last week but I still can't upload files larger than 50 MB. What should I do?",
        "assistant": "Your account upgrade can take up to 24 hours to propagate. Please try logging out and back in. If the issue persists after 24 hours, contact support at support@example.com with your account ID.",
        "latency": 1.4,         # Slower - more complex reasoning needed
        "input_tokens": 28,     # Longer question with context
        "output_tokens": 48,    # Detailed, multi-step response
        "model": "gpt-4o-mini",
    },
    {
        "name": "ambiguous_question",
        # "Delete everything" is vague - does user mean files or account?
        "user": "How do I delete everything?",
        "assistant": "You can delete all files in your workspace by going to Settings > Storage > Delete All Files. Warning: this action is irreversible. To delete your entire account, go to Settings > Account > Delete Account.",
        "latency": 1.1,         # Moderate time - handling ambiguity
        "input_tokens": 8,      # Very short, vague question
        "output_tokens": 42,    # Long response covering both interpretations
        "model": "gpt-4o-mini",
    },
    {
        "name": "out_of_scope",
        # This question has nothing to do with our product documentation
        "user": "What's the weather like in Tokyo today?",
        "assistant": "I'm a product documentation assistant and can only help with questions about our platform. For weather information, please check a weather service.",
        "latency": 0.6,         # Fast - quick rejection
        "input_tokens": 10,
        "output_tokens": 28,
        "model": "gpt-4o-mini",
    },
]

print(f"Prepared {len(QA_INTERACTIONS)} simulated interactions.")
print("\nInteraction types:")
for i, interaction in enumerate(QA_INTERACTIONS, 1):
    print(f"  {i}. {interaction['name']}: \"{interaction['user'][:50]}...\"" if len(interaction['user']) > 50 else f"  {i}. {interaction['name']}: \"{interaction['user']}\"")

Prepared 4 simulated interactions.

Interaction types:
  1. simple_factual: "What is the maximum file upload size?"
  2. multi_step_reasoning: "I upgraded to enterprise last week but I still can..."
  3. ambiguous_question: "How do I delete everything?"
  4. out_of_scope: "What's the weather like in Tokyo today?"


In [ ]:
# ============================================================================
# CREATE TRACED LLM INTERACTIONS - THE CORE OF THIS LAB
# ============================================================================
# This is where the magic happens! We create traces for each Q&A interaction.
#
# KEY CONCEPTS:
#
# 1. TRACE CREATION with start_as_current_observation():
#    - Creates a new observation (span or generation) in Langfuse
#    - The "as_type" parameter specifies what kind of observation:
#      * "span": A logical grouping of operations (like a function call)
#      * "generation": An actual LLM API call that produces text
#    - Using "with" statement ensures proper cleanup when done
#
# 2. TRACE HIERARCHY:
#    - The FIRST observation becomes the ROOT (parent) of the trace
#    - Subsequent observations created inside become CHILDREN
#    - This creates a tree structure: Trace → Root Span → Child Generation
#
# 3. WHAT GETS CAPTURED:
#    - Input: What went into this operation (the user's question)
#    - Output: What came out (the assistant's response)
#    - Metadata: Custom key-value pairs for filtering and organization
#    - Token usage: How many tokens were consumed (for cost tracking)
#    - Timing: Automatically captured start/end timestamps
#
# 4. flush():
#    - Langfuse buffers trace data for efficiency
#    - flush() forces all pending data to be sent immediately
#    - Always call flush() before querying your traces
# ============================================================================

# We'll store trace IDs so we can query these specific traces later
trace_ids = []

# Process each simulated Q&A interaction
for interaction in QA_INTERACTIONS:

    # -------------------------------------------------------------------------
    # STEP 1: Create the ROOT SPAN (parent container for this interaction)
    # -------------------------------------------------------------------------
    # This span represents the entire user interaction from start to finish.
    # Everything that happens for this request will be nested under this span.
    # The root span automatically becomes the "Trace" in Langfuse's UI.
    with langfuse.start_as_current_observation(
        as_type="span",                                    # This is a logical grouping, not an LLM call
        name=f"qa-assistant-{interaction['name']}",        # Descriptive name shown in Langfuse UI
        input=interaction["user"],                         # The user's question (trace-level input)
        metadata={
            "interaction_type": interaction["name"],       # Tag for filtering (e.g., "simple_factual")
            "source": "module-01-lab",                     # Identifies which lab created this trace
        },
    ) as root_span:

        # ---------------------------------------------------------------------
        # STEP 2: Create a CHILD GENERATION (the actual "LLM call")
        # ---------------------------------------------------------------------
        # This nested observation represents the LLM API call itself.
        # Because it's created inside the root_span's "with" block, it
        # automatically becomes a child of that span.
        with langfuse.start_as_current_observation(
            as_type="generation",                          # This represents an LLM text generation
            name="llm-completion",                         # Name for this specific LLM call
            model=interaction["model"],                    # Which model was used (gpt-4o-mini)
            input=[{"role": "user", "content": interaction["user"]}],  # OpenAI-style message format
            metadata={"simulated": True},                  # Flag that this is simulated, not a real API call
        ) as generation:

            # -----------------------------------------------------------------
            # STEP 3: Update the generation with the response and token usage
            # -----------------------------------------------------------------
            # In a real application, you'd call the LLM here and then update
            # with the actual response. We're using predefined responses.
            generation.update(
                output=interaction["assistant"],           # The assistant's response text
                usage_details={
                    "input_tokens": interaction["input_tokens"],    # Tokens in the prompt
                    "output_tokens": interaction["output_tokens"],  # Tokens in the response
                },
            )

        # ---------------------------------------------------------------------
        # STEP 4: Set the final output on the root span (trace-level output)
        # ---------------------------------------------------------------------
        # This is the "official" output of the entire interaction, visible at
        # the trace level in Langfuse's UI.
        root_span.update(output=interaction["assistant"])

        # Save the trace ID for later querying and scoring
        trace_ids.append(root_span.trace_id)

        # Print progress so we can see each trace being created
        print(f"  ✅ Traced: {interaction['name']} (trace_id: {root_span.trace_id})")

    # Simulate the realistic latency between requests
    # In production, this would be the actual API response time
    time.sleep(interaction["latency"])

# -------------------------------------------------------------------------
# STEP 5: Flush all buffered trace data to Langfuse
# -------------------------------------------------------------------------
# This ensures all traces are sent to Langfuse's servers immediately.
# Without flush(), data might sit in a buffer and not appear in the dashboard.
langfuse.flush()

# Summary message
print(f"\n🎯 All {len(trace_ids)} interactions traced and flushed to Langfuse.")
print("\n💡 You can now view these traces in your Langfuse dashboard!")

  ✅ Traced: simple_factual (trace_id: dc4aa1c9abd798f28486c29dedd998ba)
  ✅ Traced: multi_step_reasoning (trace_id: 047b6e65c72410d3432d68666bcd2e8d)
  ✅ Traced: ambiguous_question (trace_id: da64c940b8bc664caf7c36e9604b3efd)
  ✅ Traced: out_of_scope (trace_id: 3d59d18b1e458d727bba4366bd2c9bc9)

🎯 All 4 interactions traced and flushed to Langfuse.

💡 You can now view these traces in your Langfuse dashboard!


### Checkpoint ✅
You should see four `✅ Traced:` lines and finally `🎯 All 4 interactions traced and flushed to Langfuse.`

You can also open your Langfuse dashboard in a browser to confirm the traces appear there.

### Expected Output
```
  ✅ Traced: simple_factual (trace_id: ...)
  ✅ Traced: multi_step_reasoning (trace_id: ...)
  ✅ Traced: ambiguous_question (trace_id: ...)
  ✅ Traced: out_of_scope (trace_id: ...)

🎯 All 4 interactions traced and flushed to Langfuse.
```


---

## Task 4: 🔍 Think First — What Do Traces Capture That Outputs Alone Miss?

### Goal
Before diving deeper into trace data, pause and *think* about why traces matter for evaluation.

### Context
Imagine you only had access to the **outputs** of the four interactions above (the assistant's responses). Now imagine you also have the **full traces** — including latency, token usage, model info, timestamps, and metadata.

### Questions to Reason About

Answer these questions **in your own words** in the markdown cell below. Spend 2–3 minutes thinking before moving on.

1. **Latency:** Interaction #2 (multi-step reasoning) took 1.4s while Interaction #4 (out-of-scope) took 0.6s. If you only saw the outputs, would you know this? Why does latency matter for evaluation?

2. **Token usage:** Interaction #3 (ambiguous question) used only 8 input tokens but 42 output tokens. What might that ratio tell you about the interaction?

3. **Debugging:** Suppose the assistant gave a wrong answer. With only the output, what can you diagnose? With a full trace, what additional information would help you find the root cause?

4. **Patterns at scale:** If you had 10,000 traces instead of 4, what patterns could you detect from trace metadata that would be invisible from outputs alone?

> **This is a challenge section.** There are no "right" answers shown here — use your reasoning. We'll discuss as a group.

### ✍️ Your Reflections

**Write your answers here** (double-click to edit this cell):

1. **Latency:**

High spike in latency for simple queries might indicate a bacekend issue
in prod you may have latency SLA's

   *Your answer...*

2. **Token ratio:**
we see one trace with high output to input ratio - could indicate a problem ,maybe model should have asked a clarifying question instead of giving a very long answer

   *Your answer...*

3. **Debugging value:**
With tracing we can see WHY the model may have given bad answers - ie maybe wrong retrieval docs, tool mis-use, we want to see intermediate reasoning steps.
   *Your answer...*

4. **Patterns at scale:**

   *Your answer...*

---

## Task 5: Explore Trace Data Structure

### Goal
Retrieve your traces from Langfuse and examine their internal structure programmatically. Understand the hierarchy: **Trace → Observations (Generations, Spans)**.

### Steps
Run the cells below to fetch your traces and inspect what data is available.

> **SDK note:** In the current SDK, querying trace and observation data is done through the `langfuse.api` namespace.


In [ ]:
# ============================================================================
# QUERY TRACES FROM LANGFUSE - READING BACK WHAT WE SENT
# ============================================================================
# Now that we've created traces, let's retrieve them using Langfuse's API.
# This demonstrates that traces are stored remotely and can be queried by
# any system with the right credentials.
#
# WHY QUERY TRACES?
#   - Verify data was captured correctly
#   - Build evaluation pipelines that process historical traces
#   - Export data for analysis in other tools
#   - Create dashboards and reports
#
# API STRUCTURE:
#   - langfuse.api.trace.list():  Get a list of recent traces
#   - langfuse.api.trace.get():   Get a specific trace by ID
#   - langfuse.api.observations.get_many():  Get observations within a trace
#   - langfuse.api.scores.get_many():  Get scores attached to traces
# ============================================================================

import time
from httpx import ReadTimeout

# Flush any queued events before querying
langfuse.flush()
time.sleep(3)

def safe_list_traces(client, limit=5, retries=3, delay=5):
    """Fetch traces with retry logic and extended timeout."""
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            # Use request_options to set a longer timeout (60 seconds)
            return client.api.trace.list(
                limit=limit,
                request_options={"timeout_in_seconds": 60}
            )
        except ReadTimeout as e:
            last_error = e
            print(f"Attempt {attempt} timed out while fetching traces.")
            if attempt < retries:
                print(f"Waiting {delay} seconds, then retrying...\n")
                time.sleep(delay)
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed: {type(e).__name__}: {e}")
            if attempt < retries:
                print(f"Waiting {delay} seconds, then retrying...\n")
                time.sleep(delay)

    raise last_error

# Fetch traces with extended timeout
traces = safe_list_traces(langfuse, limit=5, retries=3, delay=5)

print(f"Retrieved {len(traces.data)} traces from Langfuse.\n")

print("Traces created in this lab session:")
print("-" * 60)

matched = 0

for i, trace in enumerate(traces.data, 1):
    if trace.id in trace_ids:
        matched += 1
        trace_input = trace.input if getattr(trace, "input", None) else "N/A"
        trace_output = trace.output if getattr(trace, "output", None) else "N/A"
        trace_metadata = trace.metadata if getattr(trace, "metadata", None) else {}

        print(f"\nTrace {i}: {trace.name}")
        print(f"  ID:        {trace.id}")
        print(f"  Input:     {str(trace_input)[:80]}...")
        print(f"  Output:    {str(trace_output)[:80]}...")
        print(f"  Metadata:  {trace_metadata}")

if matched == 0:
    print("No traces from this notebook run were found yet.")
    print("They may still be indexing, or your current trace_ids may be from a different run.")

Retrieved 5 traces from Langfuse.

Traces created in this lab session:
------------------------------------------------------------

Trace 5: qa-assistant-out_of_scope
  ID:        3d59d18b1e458d727bba4366bd2c9bc9
  Input:     What's the weather like in Tokyo today?...
  Output:    I'm a product documentation assistant and can only help with questions about our...
  Metadata:  {'interaction_type': 'out_of_scope', 'source': 'module-01-lab', 'resourceAttributes': {'telemetry.sdk.language': 'python', 'telemetry.sdk.name': 'opentelemetry', 'telemetry.sdk.version': '1.38.0', 'service.name': 'unknown_service'}, 'scope': {'name': 'langfuse-sdk', 'version': '4.7.0', 'attributes': {'public_key': 'pk-lf-bf1ebeec-7bbc-4066-bec2-d75d58a6ed1d'}}}


In [ ]:
# ============================================================================
# DEEP DIVE INTO A SINGLE TRACE - EXPLORING THE HIERARCHY
# ============================================================================
# Let's examine one trace in detail to understand the internal structure.
# This is crucial for:
#   - Debugging issues (why did this interaction behave this way?)
#   - Understanding the parent-child relationships between observations
#   - Seeing all the metadata and usage data captured
#
# TRACE HIERARCHY REMINDER:
#   TRACE (the overall container)
#    └── SPAN (root observation - "qa-assistant-simple_factual")
#         └── GENERATION (child observation - "llm-completion")
#
# Each observation has:
#   - id: Unique identifier
#   - type: "SPAN" or "GENERATION"
#   - name: Human-readable label
#   - input: What went into this step
#   - output: What came out
#   - metadata: Custom tags and properties
#   - usage_details: Token counts (for generations)
# ============================================================================

# Pick the first trace we created (simple_factual)
sample_trace_id = trace_ids[0]

# Fetch the trace itself (the parent container)
sample_trace = langfuse.api.trace.get(sample_trace_id)

# Fetch all observations (spans and generations) within this trace
# The 'fields' parameter specifies which data to include in the response
sample_observations = langfuse.api.observations.get_many(
    trace_id=sample_trace_id,
    limit=20,
    fields="core,basic,usage"  # Include core fields, basic info, and usage stats
)

# Display trace-level information
print(f"=== Deep Dive: {sample_trace.name} ===\n")
print(f"Trace ID:     {sample_trace.id}")
print(f"Input:        {sample_trace.input}")
print(f"Output:       {sample_trace.output}")
print(f"Metadata:     {sample_trace.metadata}")
print(f"Tags:         {sample_trace.tags}")
print(f"Observations: {len(sample_observations.data)} observation(s)\n")

# Examine each observation (both the root span and the child generation)
print("Observations within this trace:")
print("-" * 50)

for obs in sample_observations.data:
    print(f"\n  📦 Observation: {obs.name}")
    print(f"  ID:           {obs.id}")
    print(f"  Type:         {obs.type}")                          # SPAN or GENERATION
    print(f"  Model:        {getattr(obs, 'provided_model_name', None)}")  # Only generations have models
    print(f"  Input:        {obs.input}")
    print(f"  Output:       {obs.output}")
    print(f"  Usage:        {getattr(obs, 'usage_details', None)}")       # Token counts
    print(f"  Metadata:     {obs.metadata}")

=== Deep Dive: qa-assistant-simple_factual ===

Trace ID:     3e5d1307ab6129801967804f2dcbd77d
Input:        What is the maximum file upload size?
Output:       The maximum file upload size is 50 MB for free-tier users and 500 MB for enterprise users.
Metadata:     {'interaction_type': 'simple_factual', 'source': 'module-01-lab', 'resourceAttributes': {'telemetry.sdk.language': 'python', 'telemetry.sdk.name': 'opentelemetry', 'telemetry.sdk.version': '1.38.0', 'service.name': 'unknown_service'}, 'scope': {'name': 'langfuse-sdk', 'version': '4.7.0', 'attributes': {'public_key': 'pk-lf-bf1ebeec-7bbc-4066-bec2-d75d58a6ed1d'}}}
Tags:         []
Observations: 2 observation(s)

Observations within this trace:
--------------------------------------------------

  📦 Observation: llm-completion
  ID:           bd1062bf420d32c2
  Type:         GENERATION
  Model:        None
  Input:        None
  Output:       None
  Usage:        {'input_tokens': 12, 'output_tokens': 24, 'total': 36}
  Metadat

In [ ]:
# ============================================================================
# TOKEN USAGE ANALYSIS - UNDERSTANDING COSTS AND COMPLEXITY
# ============================================================================
# Token usage is one of the most valuable pieces of trace data because:
#   - LLM APIs charge per token (this directly impacts costs)
#   - Token counts indicate query complexity and response length
#   - Unusual token ratios can signal problems (e.g., overly verbose responses)
#
# In this cell, we extract and compare token usage across all our traces.
# This kind of analysis at scale helps you:
#   - Identify expensive query patterns
#   - Optimize prompts to reduce token usage
#   - Set budgets and alerts based on usage
#   - Compare efficiency across different models or prompt versions
#
# INPUT TOKENS:  Tokens in the prompt (what you send to the model)
# OUTPUT TOKENS: Tokens in the response (what the model generates)
# TOTAL TOKENS:  Sum of input + output (what you're billed for)
# ============================================================================

print("📊 Token Usage Summary Across Traces\n")
print(f"{'Trace Name':<35} {'Input Tokens':>14} {'Output Tokens':>14} {'Total':>8}")
print("-" * 75)

# Initialize totals for summary
total_input = 0
total_output = 0

# Loop through each trace we created and extract token usage from generations
for tid in trace_ids:
    # Get the trace to access its name
    trace = langfuse.api.trace.get(tid)

    # Get only GENERATION observations (these have token usage)
    # SPAN observations don't directly consume tokens
    observations = langfuse.api.observations.get_many(
        trace_id=tid,
        type="GENERATION",          # Filter to only generation observations
        limit=10,
        fields="core,basic,usage"   # Include usage statistics
    )

    # Extract and display token usage for each generation
    for obs in observations.data:
        # usage_details may be a dict or None
        usage = getattr(obs, "usage_details", None) or {}
        input_tok = usage.get("input_tokens", 0) if isinstance(usage, dict) else 0
        output_tok = usage.get("output_tokens", 0) if isinstance(usage, dict) else 0

        # Print formatted row
        print(f"{trace.name:<35} {input_tok:>14} {output_tok:>14} {input_tok + output_tok:>8}")

        # Accumulate totals
        total_input += input_tok
        total_output += output_tok

print("-" * 75)
print(f"{'TOTAL':<35} {total_input:>14} {total_output:>14} {total_input + total_output:>8}")
print(f"\n💡 In production, you'd use this data to track costs and optimize prompts.")

📊 Token Usage Summary Across Traces

Trace Name                            Input Tokens  Output Tokens    Total
---------------------------------------------------------------------------
qa-assistant-simple_factual                     12             24       36
qa-assistant-multi_step_reasoning               28             48       76
qa-assistant-ambiguous_question                  8             42       50
qa-assistant-out_of_scope                       10             28       38
---------------------------------------------------------------------------
TOTAL                                           58            142      200

💡 In production, you'd use this data to track costs and optimize prompts.


### Checkpoint ✅
You should see:
- A summary of 4 traces with their inputs, outputs, and metadata
- A detailed view of one trace showing its internal observations
- A token usage comparison table across all 4 interactions

Notice the **hierarchy**: each **Trace** contains one or more **Observations**. In this notebook, each trace has:
- a root `SPAN`
- a child `GENERATION`

In more complex applications, a trace might contain multiple spans — for retrieval, tool calls, reranking, validation, or chained LLM calls.


---

## Task 6: 🧠 Your Turn — Design Evaluation Criteria for a Traced Interaction

### Goal
Given a specific traced interaction, **you** design the evaluation criteria that should be measured. This is a core skill: deciding *what to evaluate* is often harder than running the evaluation itself.

### Context
Look at the **"ambiguous_question"** interaction below:

> **User:** "How do I delete everything?"
>
> **Assistant:** "You can delete all files in your workspace by going to Settings > Storage > Delete All Files. Warning: this action is irreversible. To delete your entire account, go to Settings > Account > Delete Account."

This response is fluent and grammatically correct. But is it *good*? That depends on what you measure.

### Your Task

In the code cell below, define **3–5 evaluation dimensions** you would use to evaluate this interaction. For each dimension, provide:

1. A short **name** (e.g., `"correctness"`)
2. A **description** of what it measures
3. A **scoring approach** — how would you score it? (e.g., binary pass/fail, 1–5 scale, LLM-as-judge)

**Hint:** Think about the multi-dimensional evaluation criteria from the module — correctness, faithfulness, robustness, safety. But also consider what's unique about *this* interaction: the query is ambiguous, the response covers a destructive action, and the user's intent is unclear.

> ⚠️ There is no single "right" answer. Define criteria that *you* think matter for a production Q&A assistant handling ambiguous, potentially dangerous requests.

In [ ]:
# ============================================================================
# 🧠 YOUR TURN: DEFINE YOUR EVALUATION CRITERIA
# ============================================================================
# This is a challenge exercise where YOU design what to evaluate.
#
# WHY THIS MATTERS:
#   Deciding WHAT to evaluate is often harder than running the evaluation.
#   There's no single "right" set of criteria — it depends on:
#     - Your application's purpose (Q&A assistant, code generator, etc.)
#     - Your users' expectations
#     - Business requirements (safety, compliance, etc.)
#     - The specific interaction being evaluated
#
# THE INTERACTION YOU'RE EVALUATING:
#   User: "How do I delete everything?"
#   Assistant: "You can delete all files in your workspace by going to
#              Settings > Storage > Delete All Files. Warning: this action
#              is irreversible. To delete your entire account, go to
#              Settings > Account > Delete Account."
#
# THINGS TO CONSIDER:
#   - The question is AMBIGUOUS (delete files? delete account?)
#   - The response discusses a DESTRUCTIVE action (irreversible deletion)
#   - The assistant covered MULTIPLE interpretations (is that good?)
#   - Did the assistant ask for CLARIFICATION (or just guessed intent)?
#
# COMMON EVALUATION DIMENSIONS:
#   - Correctness: Is the information factually accurate?
#   - Relevance: Does it answer what was asked?
#   - Helpfulness: Does it actually help the user accomplish their goal?
#   - Safety: Does it avoid harmful actions or content?
#   - Clarity: Is it easy to understand?
#   - Completeness: Does it cover all necessary information?
#   - Conciseness: Is it appropriately brief (not too long or short)?
# ============================================================================

# Replace these placeholder entries with YOUR OWN evaluation criteria
# Think about what would matter for a production Q&A assistant
my_evaluation_criteria = [
    {
        "name": "intent clarification",           # e.g., "correctness"
        "description": "did the assistant attempt to clarify the ambiguous request",    # What does this measure?
        "scoring": "0",        # How would you score it? (binary, 1-5 scale, LLM-as-judge)
    },
    {
        "name": "safety",
        "description": "is the response safe in that it warns about accidental data loss?",
        "scoring": "5",
    },
    {
        "name": "helpfulness",
        "description": "how helpful was the chatbot?",
        "scoring": "3",
    },
    # Add more criteria if you want (aim for 3-5 total)
]

# Display your criteria in a formatted way
print("📋 My Evaluation Criteria for the 'ambiguous_question' Interaction:\n")
for i, criterion in enumerate(my_evaluation_criteria, 1):
    print(f"  {i}. {criterion['name']}")
    print(f"     Description: {criterion['description']}")
    print(f"     Scoring:     {criterion['scoring']}")
    print()

print("💡 Tip: Go beyond 'is it correct?' Think about safety, clarity, and user experience.")

### Checkpoint ✅
After filling in your criteria and running the cell, you should see a formatted printout of your evaluation dimensions. Verify that:
- You have at least 3 criteria defined
- Each has a name, description, and scoring approach
- Your criteria go beyond just "is the answer correct?"

> We'll use these criteria in the next task when attaching evaluation scores to traces.

---

## Task 7: Connect Traces to Evaluation Hooks

### Goal
Attach evaluation scores to your traces using the Langfuse scoring API. This is the bridge between observability and evaluation: once scores are connected to traces, you can filter, aggregate, and trend evaluation results at scale.

### Steps
We'll score two of our traces using simple criteria. In later modules, you'll learn to automate this with LLM-as-judge and programmatic evaluation pipelines.

In [ ]:
# ============================================================================
# ATTACHING EVALUATION SCORES TO TRACES
# ============================================================================
# This is where observability meets evaluation! We're connecting evaluation
# results (scores) directly to the traces they came from.
#
# WHY ATTACH SCORES TO TRACES?
#   - Traceability: Every score links to the exact interaction it evaluated
#   - Filtering: Query traces by score (e.g., "show me all low-quality responses")
#   - Trending: Track evaluation scores over time
#   - Debugging: When something scores poorly, you have full trace context
#
# THE create_score() API:
#   - trace_id:  Which trace this score belongs to
#   - name:      The evaluation dimension (e.g., "correctness", "safety")
#   - value:     The score value (typically 0.0 to 1.0, but can be any number)
#   - comment:   Human-readable explanation of why this score was given
#
# SCORING APPROACHES:
#   - Manual: Human reviewer assigns scores (what we're doing here)
#   - Programmatic: Code-based rules (e.g., did response contain required keywords?)
#   - LLM-as-judge: Use another LLM to evaluate the response
#   - User feedback: Thumbs up/down from actual users
#
# In later modules, you'll learn to automate scoring with LLM-as-judge and
# build evaluation pipelines that score thousands of traces automatically.
# ============================================================================

# -------------------------------------------------------------------------
# SCORE TRACE 1: simple_factual
# -------------------------------------------------------------------------
# This was a straightforward question with a clear, correct answer.
# We'll score it on "correctness" and "relevance" - both should be high.

langfuse.create_score(
    trace_id=trace_ids[0],      # simple_factual trace
    name="correctness",         # Evaluation dimension
    value=1.0,                  # Score: 0.0 (wrong) to 1.0 (perfect)
    comment="Response accurately states the file upload limits for both tiers.",
)

langfuse.create_score(
    trace_id=trace_ids[0],
    name="relevance",
    value=1.0,
    comment="Directly answers the user's question without unnecessary information.",
)

print(f"✅ Scored trace: simple_factual (trace_id: {trace_ids[0]})")
print("   - correctness: 1.0")
print("   - relevance: 1.0")

# -------------------------------------------------------------------------
# SCORE TRACE 4: out_of_scope
# -------------------------------------------------------------------------
# The assistant correctly identified that weather questions are outside
# its domain and politely declined. This is GOOD behavior for a focused
# assistant - we'll score it positively on correctness and safety.

langfuse.create_score(
    trace_id=trace_ids[3],      # out_of_scope trace
    name="correctness",
    value=1.0,
    comment="Correctly identified the question as out of scope.",
)

langfuse.create_score(
    trace_id=trace_ids[3],
    name="safety",
    value=1.0,
    comment="Did not attempt to answer outside its domain; no harmful content.",
)

# Helpfulness is a bit lower - the assistant could have been more helpful
# by suggesting a specific weather service or being more graceful.
langfuse.create_score(
    trace_id=trace_ids[3],
    name="helpfulness",
    value=0.7,                  # Not perfect - could have been more helpful
    comment="Declined gracefully but could have suggested specific weather services.",
)

print(f"✅ Scored trace: out_of_scope (trace_id: {trace_ids[3]})")
print("   - correctness: 1.0")
print("   - safety: 1.0")
print("   - helpfulness: 0.7")

# -------------------------------------------------------------------------
# FLUSH SCORES TO LANGFUSE
# -------------------------------------------------------------------------
# Just like traces, scores are buffered. We flush to ensure they're saved.
langfuse.flush()

print("\n🎯 Scores attached to traces and flushed to Langfuse.")
print("\n💡 Open your Langfuse dashboard to see scores attached to each trace!")

In [ ]:
# ============================================================================
# VERIFY SCORES WERE ATTACHED - QUERYING SCORES FROM LANGFUSE
# ============================================================================
# Let's confirm the scores were properly saved by querying them back.
# This demonstrates the full loop:
#   1. Create traces (done in Task 3)
#   2. Attach scores to traces (done above)
#   3. Query scores for analysis (this cell)
#
# In production, you'd use this API to:
#   - Build dashboards showing evaluation trends
#   - Alert when scores drop below thresholds
#   - Export data for deeper analysis
#   - Compare scores across model versions or prompt variations
# ============================================================================

# Give Langfuse time to index the scores we just sent
time.sleep(5)

# Fetch the trace we scored (out_of_scope) to show in the output
scored_trace = langfuse.api.trace.get(trace_ids[3])  # out_of_scope

# Fetch all scores attached to this specific trace
trace_scores = langfuse.api.scores.get_many(trace_id=trace_ids[3])

# Display the scores in a formatted way
print(f"=== Scores for: {scored_trace.name} ===\n")

for score in trace_scores.data:
    print(f"  📏 {score.name}: {score.value}")
    print(f"     Comment: {score.comment}")
    print()

print("💡 These scores are now permanently linked to this trace in Langfuse.")
print("   You can filter, aggregate, and trend these scores in your dashboard.")

### Checkpoint ✅
You should see the scores attached to the `out_of_scope` trace:
- `correctness: 1.0`
- `safety: 1.0`
- `helpfulness: 0.7`

You can also verify in your Langfuse dashboard that traces now have scores attached.

### Expected Output
```
=== Scores for: qa-assistant-out_of_scope ===

  📏 correctness: 1.0
     Comment: Correctly identified the question as out of scope.

  📏 safety: 1.0
     Comment: Did not attempt to answer outside its domain; no harmful content.

  📏 helpfulness: 0.7
     Comment: Declined gracefully but could have suggested specific weather services.
```

---

## Validation / Success Criteria

You have completed this lab successfully if:

- [x] Langfuse connection verified (`auth_check()` passed)
- [x] 4 traced interactions created and visible in Langfuse
- [x] You explored trace data and understand the Trace → Observation hierarchy
- [x] You completed the 🔍 Think First reflection on trace value
- [x] You defined 3+ evaluation criteria in the 🧠 Your Turn challenge
- [x] Evaluation scores are attached to at least 2 traces

---

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `auth_check()` fails with 401 | Invalid API keys | Double-check `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` |
| `auth_check()` fails with connection error | Wrong host URL | Verify `LANGFUSE_BASE_URL` — EU default is `https://cloud.langfuse.com`, US is `https://us.cloud.langfuse.com` |
| `AttributeError: 'Langfuse' object has no attribute 'trace'` | Old example code on new SDK | Use `get_client()` and `start_as_current_observation(...)` |
| `AttributeError: 'Langfuse' object has no attribute 'start_trace'` | API mismatch across SDK versions | Use the current observation-based API shown in this notebook |
| Traces do not appear in dashboard yet | Data not flushed or not indexed yet | Ensure `langfuse.flush()` was called and wait 10–30 seconds |
| Trace query returns empty list | Indexing delay | Add `time.sleep(5)` before querying |
| `ModuleNotFoundError: langfuse` | Package not installed | Re-run the `pip install` cell |
| Scores do not appear | Data not flushed or query ran too early | Call `langfuse.flush()` after scoring and wait a few seconds |

---

## Stretch Goal / Extension

If you finish early, try one or more of these:

1. **Score the remaining 2 traces** (`multi_step_reasoning` and `ambiguous_question`) using the evaluation criteria you designed in Task 6.

2. **Add a multi-step trace**: Create a new trace with multiple observations — for example, a retrieval step (span) followed by an LLM generation. This simulates a RAG pipeline trace.

3. **Aggregate scores**: Write code to fetch all scored traces and compute the average score per evaluation dimension.

---

## Key Takeaways

1. **Traces capture what outputs alone miss**: latency, token usage, intermediate steps, model metadata, and the full execution path.

2. **Langfuse makes tracing straightforward**: a few lines of code give you structured observability for LLM applications.

3. **Trace-based evaluation scales better than intuition**: once traces and scores are linked, you can filter, compare, and monitor quality over time.


# Is Langfuse free?

Yes — **Langfuse *can* be free**, but it depends on how you use it.

### 1) Free cloud (managed by Langfuse)

* There’s a **free “Hobby” tier**
* Typically includes:

  * ~**50,000 units/month** (traces, spans, evals, etc.) ([Leanware][1])
  * **~30-day data retention** ([Leanware][1])
  * Limited users (often ~2 users) ([Leanware][1])
* Once you exceed limits → you need to upgrade (no overage on free tier) ([CheckThat.ai][2])

👉 This is what most people use for quick experiments or small projects.

---

### 2) 100% free (self-hosted)

* Langfuse is **open-source (MIT license)**
* You can **run it yourself (Docker, etc.) with no platform cost** ([Langfuse][3])
* No enforced usage limits from Langfuse itself (just your infra limits) ([GitHub][4])

👉 This is what teams use for:

* enterprise data control
* scaling cheaply
* avoiding SaaS pricing

---

### 3) Paid plans (when you scale)

* Paid tiers start around:

  * ~$29/month (Core)
  * ~$199/month (Pro) ([Confident AI][5])
* Pricing is **usage-based (units)**, not per seat (which is nice for teams) ([CheckThat.ai][2])

---

### Bottom line

* **Yes — free to start (cloud tier)**
* **Yes — completely free if self-hosted**
* You only pay when:

  * you exceed free usage
  * or want managed hosting + advanced features

---

For our use case  (labs / Colab):
👉 The **free cloud tier OR self-hosting** is usually more than enough.

[1]: https://www.leanware.co/insights/langfuse-vs-langsmith?utm_source=chatgpt.com "Langfuse vs LangSmith: Full Comparison"
[2]: https://checkthat.ai/brands/langfuse/pricing?utm_source=chatgpt.com "Langfuse Pricing 2026: Plans, Costs & Breakdown"
[3]: https://langfuse.com/pricing-self-host?utm_source=chatgpt.com "Pricing Self Host"
[4]: https://github.com/orgs/langfuse/discussions/10329?utm_source=chatgpt.com "Limits Free Tier self-hosted · langfuse · Discussion #10329"
[5]: https://www.confident-ai.com/knowledge-base/top-7-llm-observability-tools?utm_source=chatgpt.com "Top 7 LLM Observability Tools in 2026"
